# Context Window Experiments

This notebook explores how conversation memory and context size affect an AI chatbot.

## Experiments

### 1. Conversation Memory
We compare three memory strategies:

- Last 5 messages
- Last 20 messages
- Entire conversation history

### 2. Context Overflow
We gradually increase the amount of text given to the model and observe:

- Token count
- Response latency
- Response quality

## Goal

Understand how much conversation history an AI system can keep,
how context size affects performance, and why context-window management
is important in real-world LLM applications.

In [1]:
import tiktoken
import time
import pandas as pd

In [2]:
encoding = tiktoken.get_encoding('o200k_base')

In [3]:
text = ' how are you maria?'
tokens = encoding.encode(text)
print('the tokens are: ', tokens)

print('token count: ',len(tokens))

the tokens are:  [1495, 553, 481, 172331, 30]
token count:  5


In [4]:
def count_tokens(text):
    tokens = encoding.encode(text)
    return len(tokens)

In [5]:
print(len(tokens))

5


In [6]:
text = 'the autofocus is working perfectly fine without any error'
print('text: ',text)
print('token count: ', count_tokens(text))

text:  the autofocus is working perfectly fine without any error
token count:  9


## Experiment 1: Conversation Memory

A chatbot can store previous messages as conversation history.

However, sending the entire history every time can make the context very large.

We will compare three strategies:

1. Keep the last 5 messages
2. Keep the last 20 messages
3. Keep the entire conversation

The goal is to understand what information is retained and what information is lost.

In [7]:
conversation = [
    {"role": "user", "content": "My name is Rahul."},
    {"role": "assistant", "content": "Nice to meet you, Rahul!"},
    
    {"role": "user", "content": "I am building an ESP32 project."},
    {"role": "assistant", "content": "That sounds interesting. What does it do?"},
    
    {"role": "user", "content": "It controls an autofocus motor."},
    {"role": "assistant", "content": "So the ESP32 controls the motor movement."},
    
    {"role": "user", "content": "The motor communicates using SPI."},
    {"role": "assistant", "content": "Got it. SPI is being used for communication."},
    
    {"role": "user", "content": "Sometimes the SPI communication times out."},
    {"role": "assistant", "content": "Then the motor may stop responding correctly."},
    
    {"role": "user", "content": "I added logging to debug the problem."},
    {"role": "assistant", "content": "Logging should help identify when the timeout happens."},
    
    {"role": "user", "content": "The system works for about 20 cycles."},
    {"role": "assistant", "content": "That suggests the problem may appear after repeated operation."},
    
    {"role": "user", "content": "After that, the motor sometimes stops."},
    {"role": "assistant", "content": "We should investigate what changes after those cycles."},
    
    {"role": "user", "content": "I also checked the power supply."},
    {"role": "assistant", "content": "Good idea. Power instability can affect motor communication."},
    
    {"role": "user", "content": "The voltage looks stable."},
    {"role": "assistant", "content": "Then we can focus more on the software and communication."},
    
    {"role": "user", "content": "I want to understand the root cause."},
    {"role": "assistant", "content": "We can analyze the logs and communication sequence."},
    
    {"role": "user", "content": "The important part is that it only happens after many cycles."},
    {"role": "assistant", "content": "That makes repeated-state or resource-related issues worth investigating."},
    
    {"role": "user", "content": "I am using FreeRTOS tasks."},
    {"role": "assistant", "content": "Then task scheduling and synchronization could also be relevant."},
    
    {"role": "user", "content": "One task handles the motor control."},
    {"role": "assistant", "content": "We should also check how that task interacts with the SPI task."},
    
    {"role": "user", "content": "I want to test whether memory affects the chatbot."},
    {"role": "assistant", "content": "Let's compare different conversation-memory strategies."},
]

In [8]:
print("Total messages:", len(conversation))

Total messages: 30


In [9]:
for message in conversation: 
    print(message['role'],':', message['content'])

user : My name is Rahul.
assistant : Nice to meet you, Rahul!
user : I am building an ESP32 project.
assistant : That sounds interesting. What does it do?
user : It controls an autofocus motor.
assistant : So the ESP32 controls the motor movement.
user : The motor communicates using SPI.
assistant : Got it. SPI is being used for communication.
user : Sometimes the SPI communication times out.
assistant : Then the motor may stop responding correctly.
user : I added logging to debug the problem.
assistant : Logging should help identify when the timeout happens.
user : The system works for about 20 cycles.
assistant : That suggests the problem may appear after repeated operation.
user : After that, the motor sometimes stops.
assistant : We should investigate what changes after those cycles.
user : I also checked the power supply.
assistant : Good idea. Power instability can affect motor communication.
user : The voltage looks stable.
assistant : Then we can focus more on the software and 

In [10]:
last_5 = conversation[-5:]
first_5 = conversation[ :5]
last_20 = conversation[-20:]
entire_history = conversation
print('the last 5 msgs are: ',last_5)
print('the first 5 msgs are: ',first_5)
print('the last 20 msgs are: ',last_20)

the last 5 msgs are:  [{'role': 'assistant', 'content': 'Then task scheduling and synchronization could also be relevant.'}, {'role': 'user', 'content': 'One task handles the motor control.'}, {'role': 'assistant', 'content': 'We should also check how that task interacts with the SPI task.'}, {'role': 'user', 'content': 'I want to test whether memory affects the chatbot.'}, {'role': 'assistant', 'content': "Let's compare different conversation-memory strategies."}]
the first 5 msgs are:  [{'role': 'user', 'content': 'My name is Rahul.'}, {'role': 'assistant', 'content': 'Nice to meet you, Rahul!'}, {'role': 'user', 'content': 'I am building an ESP32 project.'}, {'role': 'assistant', 'content': 'That sounds interesting. What does it do?'}, {'role': 'user', 'content': 'It controls an autofocus motor.'}]
the last 20 msgs are:  [{'role': 'user', 'content': 'I added logging to debug the problem.'}, {'role': 'assistant', 'content': 'Logging should help identify when the timeout happens.'}, {

In [11]:
memory_comparison = pd.DataFrame({
    'memory strategy': ['last 5 msgs', ' first 5 msgs', 'last 20 msgs','entire history'],
    'msgs kept': [len(last_5), len(first_5), len(last_20), len(entire_history)]
})
memory_comparison

,memory strategy,msgs kept
0,last 5 msgs,5
1,first 5 msgs,5
2,last 20 msgs,20
3,entire history,30


In [12]:
last_5_text = ""

for message in last_5:
    last_5_text = last_5_text + message["content"] + " "

last_20_text = ""

for message in last_20:
    last_20_text = last_20_text + message["content"] + " "

entire_text = ""

for message in entire_history:
    entire_text = entire_text + message["content"] + " "


last_5_tokens = len(encoding.encode(last_5_text))
last_20_tokens = len(encoding.encode(last_20_text))
entire_tokens = len(encoding.encode(entire_text))


print("Last 5 messages:", last_5_tokens, "tokens")
print("Last 20 messages:", last_20_tokens, "tokens")
print("Entire history:", entire_tokens, "tokens")

Last 5 messages: 48 tokens
Last 20 messages: 182 tokens
Entire history: 257 tokens


In [13]:
memory_comparison = pd.DataFrame({
    "Memory Strategy": ["Last 5", "Last 20", "Entire History"],
    "Messages": [5, 20, 30],
    "Tokens": [last_5_tokens, last_20_tokens, entire_tokens]
})

memory_comparison

,Memory Strategy,Messages,Tokens
0,Last 5,5,48
1,Last 20,20,182
2,Entire History,30,257


In [14]:
comparison_table = pd.DataFrame({
    'msgs stored': ["last 5","last 20","entire history"],
    'no. of msgs': [5,20,30],
    'tokens': [last_5_tokens, last_20_tokens, entire_tokens]
    })

comparison_table

,msgs stored,no. of msgs,tokens
0,last 5,5,48
1,last 20,20,182
2,entire history,30,257


## Experiment 2: Context Overflow

In this experiment, we gradually increase the amount of text given to the system.

We will measure how the context size changes as more text is added.

The main measurement is token count.

In [15]:
base_text = """
The ESP32 controls an autofocus motor.
The motor communicates using SPI.
Sometimes the SPI communication times out.
The system works correctly for several cycles.
After repeated operation, the motor may stop responding.
We are checking logs to understand the root cause.
"""

In [16]:
small_text = base_text *1
medium_text = base_text*5
large_text = base_text* 10
very_large_text = base_text * 20


In [17]:
text = 'hello '
print(text * 5)

hello hello hello hello hello 


In [18]:
small_tokens = len(encoding.encode(small_text))
medium_tokens = len(encoding.encode(medium_text))
large_tokens = len(encoding.encode(large_text))
very_large_tokens = len(encoding.encode(very_large_text))

print("Small:", small_tokens, "tokens")
print("Medium:", medium_tokens, "tokens")
print("Large:", large_tokens, "tokens")
print("Very Large:", very_large_tokens, "tokens")

Small: 50 tokens
Medium: 246 tokens
Large: 491 tokens
Very Large: 981 tokens


In [19]:
token_table = pd.DataFrame({
    'text_size' : ['small','medium','large','verylarge']  ,
    'token_count': [50,246,491,981]  
})
token_table

,text_size,token_count
0,small,50
1,medium,246
2,large,491
3,verylarge,981


In [20]:
start_time = time.time()
tokens = encoding.encode(small_text)
end_time = time.time()
latency = end_time - start_time 
print('latency', latency, 'seconds')

latency 0.0002117156982421875 seconds


In [33]:
start_time_1 = time.time()
tokens_1 = encoding.encode(small_text)
end_time_1 = time.time()
latency_1 = end_time_1 - start_time_1 
print('latency of small text is : ', latency_1, 'seconds')

start_time_2 = time.time()
tokens_2 = encoding.encode(large_text)
end_time_2 = time.time()
latency_2 = end_time_2 - start_time_2
print('latency of large text is : ', latency_2, 'seconds')

start_time_3 = time.time()
tokens_3 = encoding.encode(medium_text)
end_time_3 = time.time()
latency_3 = end_time_3 - start_time_3
print('latency of medium text is: ', latency_3,'seconds')

start_time = time.time()
encoding.encode(very_large_text)
end_time = time.time()
latency_4 = end_time-start_time
print('latency of very large text is: ', latency_4, 'seconds')

latency of small text is :  0.00028395652770996094 seconds
latency of large text is :  0.00026917457580566406 seconds
latency of medium text is:  0.0003943443298339844 seconds
latency of very large text is:  0.00029850006103515625 seconds


In [34]:
token_table = pd.DataFrame({
    'text size' : ['small','medium','large','very_large'],
    'token count': [50,246,491,981],
    'latency': [latency_1,latency_2,latency_3,latency_4]
})

token_table

,text size,token count,latency
0,small,50,0.000284
1,medium,246,0.000269
2,large,491,0.000394
3,very_large,981,0.000299


In [28]:
start_time = time.time()
encoding.encode(small_text)
small_latency = time.time() - start_time

start_time = time.time()
encoding.encode(medium_text)
medium_latency = time.time() - start_time

start_time = time.time()
encoding.encode(large_text)
large_latency = time.time() - start_time

start_time = time.time()
encoding.encode(very_large_text)
very_large_latency = time.time() - start_time

print("Small:", small_latency, "seconds")
print("Medium:", medium_latency, "seconds")
print("Large:", large_latency, "seconds")
print("Very Large:", very_large_latency, "seconds")

Small: 0.000152587890625 seconds
Medium: 0.0004761219024658203 seconds
Large: 0.00027823448181152344 seconds
Very Large: 0.0007026195526123047 seconds


## Response Quality

As the context becomes larger, important information may become harder for a model to use effectively.

We will test whether an important fact is still available in each context size.

In [36]:
question = 'how many cycles does it take for the system to work correctly?'
expected_answer = '20 cycles'

print('question: ',question)
print('answer: ', expected_answer)

question:  how many cycles does it take for the system to work correctly?
answer:  20 cycles


In [37]:
base_text = """
The ESP32 controls an autofocus motor.
The motor communicates using SPI.
Sometimes the SPI communication times out.
The system works correctly for about 20 cycles.
After repeated operation, the motor may stop responding.
We are checking logs to understand the root cause.
"""

In [39]:
small_quality = expected_answer in small_text
medium_quality = expected_answer in medium_text
large_quality = expected_answer in large_text
very_large_quality = expected_answer in very_large_text

print("Small context:", small_quality)
print("Medium context:", medium_quality)
print("Large context:", large_quality)
print("Very large context:", very_large_quality)


Small context: False
Medium context: False
Large context: False
Very large context: False


### Limitation

This notebook does not currently call an external LLM.

Therefore, the latency measurements represent local tokenization time rather than actual LLM response latency.

Similarly, the response-quality test checks whether the expected information is present in the context. It does not measure the quality of an actual generated LLM response.

A future version could connect this experiment to an LLM API and measure real response latency and answer quality.

In [40]:
final_results = pd.DataFrame({
    "Context Size": ["Small", "Medium", "Large", "Very Large"],
    "Token Count": [
        small_tokens,
        medium_tokens,
        large_tokens,
        very_large_tokens
    ],
    "Latency (seconds)": [
        small_latency,
        medium_latency,
        large_latency,
        very_large_latency
    ],
    "Information Available": [
        small_quality,
        medium_quality,
        large_quality,
        very_large_quality
    ]
})

final_results

,Context Size,Token Count,Latency (seconds),Information Available
0,Small,50,0.000153,False
1,Medium,246,0.000476,False
2,Large,491,0.000278,False
3,Very Large,981,0.000703,False


## Final Observations

### Conversation Memory

Keeping only the last few messages reduces the number of tokens sent to the model, but older information may no longer be available in the context.

Keeping more messages preserves more conversation information but increases the context size.

### Context Size

As more text is added, the token count increases.

Larger contexts require the system to process more information.

### Response Quality

In this experiment, the important information remained available even as the context increased.

However, this test does not measure actual LLM-generated response quality because no external LLM was used.

### Key Learning

There is a trade-off between:

**More memory → more information → more tokens**

and

**Less memory → fewer tokens → less historical information**

Real chatbot systems therefore need techniques such as truncation, summarization, retrieval, and other context-management strategies to control context size.